# Phase 1: Preprocessing and Model Training

## 1. Raw Data Preprocessing

Filters, resamples, and prepares the raw iEEG recording for manual cleaning.
Depth channels are band-pass filtered at 0.1–499 Hz;
zEEG channels are band-pass filtered at 0.1–40 Hz.
A notch filter removes line noise and harmonics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, precision_recall_curve, auc
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
import joblib
import mne
from zeeg_utils import raw_chan_to_feat

In [ ]:
subj = 'P01'
raw_file = f'{subj}_overnightData.fif'

zeeg = ['zeeg1', 'zeeg2']
depth = sorted(['RAH1', 'LAH1', 'RA1', 'LA1', 'LEC1', 'REC1',
                 'RPHG1', 'LPHG1', 'RMH1', 'LMH1',
                 'RAH2', 'LAH2', 'RA2', 'LA2', 'LEC2', 'REC2',
                 'RPHG2', 'LPHG2', 'RMH2', 'LMH2'])
notch = (50, 100, 150, 200, 250)

In [ ]:
raw = mne.io.read_raw(raw_file, preload=False)

curr_zeeg = [x for x in zeeg if x in raw.ch_names]
curr_depth = [x for x in depth if x in raw.ch_names]
raw.pick(curr_depth + curr_zeeg)
raw.load_data()
raw.resample(1000)
raw.filter(l_freq=0.1, h_freq=499, picks=curr_depth, phase='zero-double')
raw.notch_filter(notch, method='spectrum_fit', phase='zero-double')
raw.filter(l_freq=0.1, h_freq=40, picks=curr_zeeg, phase='zero-double')

### Manual Cleaning

Visually inspect the filtered data, mark bad segments and bad channels,
then save the clean file.

In [ ]:
raw.plot(duration=60*5, block=True)

In [ ]:
raw.drop_channels(raw.info['bads'], on_missing='ignore')
raw.save(f'{subj}_mtl_clean.fif', overwrite=True)

## 2. Feature Extraction: iEEG + zEEG Training Data

Loads cleaned iEEG recordings, extracts depth electrode features
to generate ED labels, and extracts zEEG features for model training.

In [ ]:
sr = 1000  # Hz
window_size = 250 # samples
mtl_path = 'P%s_mtl_clean.fif'
all_subjects = ['01', '02', '03'] # example ids
depth_model = joblib.load('depth_model.pkl')
depth_channels = ['RAH1', 'LAH1', 'RA1', 'LA1', 'LEC1', 'REC1', 'RPHG1', 'LPHG1', 'RMH1', 'LMH1', 'RAH2', 'LAH2', 'RA2', 'LA2', 'LEC2', 'REC2', 'RPHG2', 'LPHG2', 'RMH2', 'LMH2']

In [ ]:
def get_depth_pred(subjects, threshold=0.8, min_channels=2):
    """
    Generates depth model predictions (labels) for a list of subjects.

    This function iterates through subjects, loads data, extracts features
    from target channels, and generates a binary prediction for each epoch
    based on a consensus of channels.

    Args:
        subjects: List of subject IDs.
        threshold (float): Probability threshold for a single channel to be considered "active".
        min_channels (int): The minimum number of active channels required to
                            label an epoch as positive (1).

    Returns:
        dict: A dictionary where keys are subject IDs and values are np.ndarrays
              of binary predictions (one per epoch).
    """
    y_all = {}
    for subj in subjects:
        raw = mne.io.read_raw(mtl_path % subj)
        curr_chans = [chan for chan in raw.ch_names if chan in depth_channels]
        y_curr = None
        for chan in curr_chans:
            curr_feat = raw_chan_to_feat(raw, chan, subj, depth=True)
            predictions = depth_model.predict_proba(curr_feat[depth_model.get_booster().feature_names])
            if y_curr is None:
                y_curr = (predictions[:, 1] >= threshold).astype(int)
            else:
                y_curr += (predictions[:, 1] >= threshold).astype(int)

        y_curr[y_curr <= min_channels - 1] = 0
        y_curr[y_curr > min_channels - 1] = 1
        y_all[subj] = y_curr

    return y_all

In [ ]:
def get_all_features_per_chan(chan, subjects):
    """
    Runs feature extraction for a single channel across all subjects.

    Args:
        chan (str): The name of the channel to extract features for (e.g., "zeeg1").
        subjects (List[str]): List of subject IDs to process.

    Returns:
        dict: A dictionary where keys are subject IDs and values are the
              feature DataFrames returned by raw_chan_to_feat.
    """
    all_features = {}
    for subj in subjects:
        raw = mne.io.read_raw(mtl_path % subj)
        curr_feat = raw_chan_to_feat(raw, chan, subj, depth=False)
        all_features[subj] = curr_feat

    return all_features

In [ ]:
# Run feature extraction and save training data
y_all = get_depth_pred(all_subjects)
zeeg1_all = get_all_features_per_chan('zeeg1', all_subjects)
zeeg2_all = get_all_features_per_chan('zeeg2', all_subjects)
subj_data = {}
for subj in all_subjects:
    subj_data[subj] = {'zeeg1': zeeg1_all[subj], 'zeeg2': zeeg2_all[subj], 'y': y_all[subj]}

joblib.dump(subj_data, 'zeeg_training_data.pkl')

## 3. Model Training and Evaluation

Train an XGBoost classifier using 5-fold cross-validation on balanced,
symmetrically augmented zEEG features with depth-derived ED labels.

In [ ]:
symmetric = True
max_samples = 3000

In [ ]:
subj_data = joblib.load('zeeg_training_data.pkl')
# combine all subjects
x = pd.DataFrame()
y = np.array([])
for subj in all_subjects:
    zeeg1_subj = subj_data[subj]['zeeg1']
    zeeg2_subj = subj_data[subj]['zeeg2']
    y_subj = subj_data[subj]['y']
    zeeg1_subj.reset_index(drop=True, inplace=True)
    zeeg2_subj.reset_index(drop=True, inplace=True)
    x_subj = pd.concat([zeeg1_subj, zeeg2_subj], axis=1, ignore_index=True)
    x_subj.columns = [f'zeeg1_{col}' for col in zeeg1_subj.columns] + [f'zeeg2_{col}' for col in zeeg2_subj.columns]
    x = pd.concat([x, x_subj], ignore_index=True)
    if symmetric:
        x_sym = pd.concat([zeeg2_subj, zeeg1_subj], axis=1, ignore_index=True)
        x_sym.columns = x_subj.columns
        x = pd.concat([x, x_sym], ignore_index=True)
        y = np.concatenate((y, y_subj))
    y = np.concatenate((y, y_subj))

In [ ]:
# Balance data per subject
x['pred'] = y
sampled_data_0 = pd.DataFrame()
sampled_data_1 = pd.DataFrame()
for subj in all_subjects:
    n_EDs = x[(x['zeeg1_subj'] == subj) & (x['pred'] == 1)].shape[0]
    sample_count = min(max_samples, n_EDs)
    sampled_data_0 = pd.concat([sampled_data_0, x[(x['zeeg1_subj'] == subj) & (x['pred'] == 0)].sample(sample_count, replace=True, random_state=8)])
    sampled_data_1 = pd.concat([sampled_data_1, x[(x['zeeg1_subj'] == subj) & (x['pred'] == 1)].sample(sample_count, replace=True, random_state=8)])

sampled_data = pd.concat([sampled_data_1, sampled_data_0], ignore_index=True)
x = sampled_data.drop(columns='pred')
y = sampled_data['pred']
sampled_data

In [ ]:
# 5-fold stratified Cross-Validation
meta_data = ['subj', 'epoch_id', 'chan_name', 'epoch']
x_feat = x[x.columns[~x.columns.str.contains('|'.join(meta_data))]]

metrics = {'accuracy': [], 'precision': [], 'sensitivity': [], 'specificity': [], 'f1': [], 'ROCAUC': [], 'PRAUC': []}
all_y_true = []
all_y_prob = []

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=8)
for train_index, test_index in kf.split(x_feat, y):
    model = xgb.XGBClassifier() 
    x_train_fold, x_test_fold = x_feat.iloc[train_index], x_feat.iloc[test_index]
    y_train_fold, y_test_fold = y[train_index], y[test_index]
    
    model.fit(x_train_fold, y_train_fold)
    y_prob = model.predict_proba(x_test_fold)[:, 1]
    y_pred = (y_prob > 0.5).astype(int)
    y_true = y_test_fold
    all_y_true.extend(y_true)
    all_y_prob.extend(y_prob)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    metrics['accuracy'].append(accuracy_score(y_true, y_pred))
    metrics['precision'].append(precision_score(y_true, y_pred))
    metrics['sensitivity'].append(recall_score(y_true, y_pred))
    metrics['f1'].append(f1_score(y_true, y_pred))
    metrics['specificity'].append(tn / (tn + fp))
    metrics['ROCAUC'].append(roc_auc_score(y_true, y_prob))
    metrics['PRAUC'].append(average_precision_score(y_true, y_prob))

results = pd.DataFrame(metrics)
results.loc['mean'] = results.mean()
results

In [ ]:
# Plot ROC and PR curves
fpr, tpr, _ = roc_curve(all_y_true, all_y_prob)
precision, recall, _ = precision_recall_curve(all_y_true, all_y_prob)
roc_auc = auc(fpr, tpr)
pr_auc = auc(recall, precision)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(recall, precision, label=f'PR curve (AUC = {pr_auc:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.show()